In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats # For Z-score

# Load data
df = pd.read_csv("../data/sudan.csv") # Repeat for each country
df['Country'] = 'Sudan'

# Date Parsing
# The logic: YEAR * 1000 + DOY creates a format like 2015032
df['date'] = pd.to_datetime(df['YEAR'] * 1000 + df['DOY'], format='%Y%j')
df['Month'] = df['date'].dt.month

In [ ]:
# 1. Handle Sentinels
df.replace(-999, np.nan, inplace=True)

# 2. Drop Duplicates
initial_count = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_count - len(df)} duplicate rows.")

# 3. Missing Value Analysis
missing_percentage = df.isna().mean() * 100
print(missing_percentage[missing_percentage > 5])

In [ ]:
cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_scores = np.abs(stats.zscore(df[cols_to_check], nan_policy='omit'))

# Flag rows where any value has |Z| > 3
outliers = (z_scores > 3).any(axis=1)
print(f"Number of rows with outliers: {outliers.sum()}")

# Decide: Cap, drop, or keep? Document your choice in Markdown.

In [ ]:
# Monthly Mean Temp
monthly_t2m = df.groupby('Month')['T2M'].mean()
monthly_t2m.plot(kind='line', marker='o')
plt.title('Average Monthly Temperature')
plt.show()

# Monthly Total Precipitation
df.groupby('Month')['PRECTOTCORR'].sum().plot(kind='bar')
plt.title('Total Monthly Precipitation')
plt.show()

In [ ]:

# Heatmap - using numeric_only=True to ignore the Country/Date columns
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

# Bubble Chart: T2M vs RH2M with Bubble Size = Precipitation
plt.scatter(df['T2M'], df['RH2M'], s=df['PRECTOTCORR']*5, alpha=0.5)
plt.xlabel('Temperature')
plt.ylabel('Relative Humidity')
plt.show()